In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import xgboost as xgb
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import lightgbm as lgb

In [10]:
df_old = pd.read_csv(r'data/permits_old.csv', low_memory=False, encoding='ISO-8859-1')
df_new = pd.read_csv(r'data/permits_new.csv', low_memory=False, encoding='ISO-8859-1')
pd.set_option('display.max_columns', None)
print(df_new.head())
print(df_old.head())

   _id     PERMIT_NUM REVISION_NUM             PERMIT_TYPE  \
0    1  19 146184 DRN           00  Drain and Site Service   
1    2  19 176553 DRN           00  Drain and Site Service   
2    3  19 176561 DRN           00  Drain and Site Service   
3    4  19 160209 DRN           00  Drain and Site Service   
4    5  19 160397 DRN           00  Drain and Site Service   

        STRUCTURE_TYPE                          WORK STREET_NUM STREET_NAME  \
0  SFD - Semi-Detached  Building Permit Related (DR)         11      FULTON   
1       SFD - Detached  Building Permit Related (DR)          6     ARDMORE   
2       SFD - Detached  Building Permit Related (DR)         92   DUNINGTON   
3       SFD - Detached  Building Permit Related (DR)         96    WANSTEAD   
4       SFD - Detached  Building Permit Related (DR)        392     DOUGLAS   

  STREET_TYPE STREET_DIRECTION POSTAL     GEO_ID WARD_GRID APPLICATION_DATE  \
0         AVE                     M4K   806715.0     S1424       2019-04-

In [11]:
float_cols = df_new.select_dtypes(include=['float64']).columns
for col in float_cols:
    print(f"Column: {col}")
    print("Null values:", df_new[col].isnull().sum())
    print("NaN values:", df_new[col].isna().sum())
    print("Zero values:", (df_new[col] == 0).sum())
    print("\n")

Column: GEO_ID
Null values: 7230
NaN values: 7230
Zero values: 0


Column: DWELLING_UNITS_CREATED
Null values: 242462
NaN values: 242462
Zero values: 109743


Column: DWELLING_UNITS_LOST
Null values: 242787
NaN values: 242787
Zero values: 123690


Column: ASSEMBLY
Null values: 0
NaN values: 0
Zero values: 374381


Column: INSTITUTIONAL
Null values: 0
NaN values: 0
Zero values: 375130


Column: RESIDENTIAL
Null values: 0
NaN values: 0
Zero values: 342748


Column: BUSINESS_AND_PERSONAL_SERVICES
Null values: 0
NaN values: 0
Zero values: 374653


Column: MERCANTILE
Null values: 0
NaN values: 0
Zero values: 374574


Column: INDUSTRIAL
Null values: 0
NaN values: 0
Zero values: 374152


Column: INTERIOR_ALTERATIONS
Null values: 1
NaN values: 1
Zero values: 320457


Column: DEMOLITION
Null values: 0
NaN values: 0
Zero values: 363589




In [12]:
df_new = df_new.drop(columns=['_id', 'BUILDER_NAME', 'DEMOLITION', 'STREET_DIRECTION', 'STREET_NUM', 'STREET_NAME', 'STREET_TYPE'])
df_old = df_old.drop(columns=['STREET_DIRECTION', 'STREET_NUM', 'STREET_NAME', 'STREET_TYPE'])
print(df_new.info())
print(df_old.info())

<class 'pandas.DataFrame'>
RangeIndex: 375163 entries, 0 to 375162
Data columns (total 25 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   PERMIT_NUM                      375163 non-null  str    
 1   REVISION_NUM                    375163 non-null  str    
 2   PERMIT_TYPE                     375163 non-null  str    
 3   STRUCTURE_TYPE                  373084 non-null  str    
 4   WORK                            375108 non-null  str    
 5   POSTAL                          375163 non-null  str    
 6   GEO_ID                          367933 non-null  float64
 7   WARD_GRID                       375158 non-null  str    
 8   APPLICATION_DATE                375157 non-null  str    
 9   ISSUED_DATE                     343994 non-null  str    
 10  COMPLETED_DATE                  375163 non-null  str    
 11  STATUS                          375163 non-null  str    
 12  DESCRIPTION                

In [13]:
df_new.columns = df_new.columns.str.lower()
df_old.columns = df_old.columns.str.lower()
print(df_new.info())
print(df_old.info())

<class 'pandas.DataFrame'>
RangeIndex: 375163 entries, 0 to 375162
Data columns (total 25 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   permit_num                      375163 non-null  str    
 1   revision_num                    375163 non-null  str    
 2   permit_type                     375163 non-null  str    
 3   structure_type                  373084 non-null  str    
 4   work                            375108 non-null  str    
 5   postal                          375163 non-null  str    
 6   geo_id                          367933 non-null  float64
 7   ward_grid                       375158 non-null  str    
 8   application_date                375157 non-null  str    
 9   issued_date                     343994 non-null  str    
 10  completed_date                  375163 non-null  str    
 11  status                          375163 non-null  str    
 12  description                

In [14]:
#Combine the two datasets, keeping only the common columns
common_columns = list(set(df_new.columns) & set(df_old.columns))
df_permits = pd.concat([df_new[common_columns], df_old[common_columns]], ignore_index=True)
print(df_permits.info())

<class 'pandas.DataFrame'>
RangeIndex: 823256 entries, 0 to 823255
Data columns (total 25 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   institutional                   823256 non-null  float64
 1   structure_type                  820197 non-null  str    
 2   geo_id                          794943 non-null  float64
 3   permit_type                     823256 non-null  str    
 4   postal                          823256 non-null  str    
 5   revision_num                    823256 non-null  str    
 6   permit_num                      823256 non-null  str    
 7   application_date                823217 non-null  str    
 8   description                     821100 non-null  str    
 9   dwelling_units_lost             272264 non-null  float64
 10  proposed_use                    629472 non-null  str    
 11  residential                     823256 non-null  float64
 12  mercantile                 

In [15]:
# Function to safely parse your specific timestamp format
def robust_date_parse(series):
    # 1. Force to string and clean whitespace
    s = series.astype(str).str.strip()
    
    # 2. Remove the "Garbage" text if it exists here too
    s = s.replace("DO NOT UPDATE OR DELETE THIS INFO FIELD", pd.NA)
    s = s.replace('nan', pd.NA) # Handle string 'nan'
    
    # 3. Parse with explicit format (Much faster)
    # format='mixed' allows Pandas to handle both '2000-01-01' and '2000-01-01 12:00:00.000'
    return pd.to_datetime(s, format='mixed', errors='coerce')

# Apply to your columns
print("Parsing Dates...")
df_permits['application_date'] = robust_date_parse(df_permits['application_date'])
df_permits['issued_date'] = robust_date_parse(df_permits['issued_date'])
df_permits['completed_date'] = robust_date_parse(df_permits['completed_date'])

# Check the result
print(df_permits[['application_date', 'issued_date', 'completed_date']].info())
print(df_permits[['application_date', 'issued_date', 'completed_date']].head())

Parsing Dates...
<class 'pandas.DataFrame'>
RangeIndex: 823256 entries, 0 to 823255
Data columns (total 3 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   application_date  823217 non-null  datetime64[us]
 1   issued_date       775397 non-null  datetime64[us]
 2   completed_date    823256 non-null  datetime64[us]
dtypes: datetime64[us](3)
memory usage: 18.8 MB
None
  application_date issued_date completed_date
0       2019-04-29  2019-06-13     2020-04-14
1       2019-06-23  2019-07-09     2025-07-11
2       2019-06-23  2019-07-10     2020-07-14
3       2019-05-27  2019-07-09     2022-01-28
4       2019-05-27  2019-06-19     2021-01-11


In [16]:
#Checkpoint, save the combined dataset to a new CSV file
df_permits.to_csv('data/permits_combined.csv', index=False)